# ONNX & ONNX Runtime

A refresher on **ONNX** (the model file format) and **ONNX Runtime / `onnxruntime`** (the engine that runs it). ONNX is the "PDF of neural networks": you train in one framework, export to a single portable file, and run it anywhere — server, browser, phone, edge — usually faster than the source framework.

**Domain:** AI/ML Tooling  ·  **recommended addition**  ·  **runnable:** yes

## 1. What & Why

**What it is.** Two separate things that share a name:

- **ONNX (Open Neural Network Exchange)** — an *open file format* (`.onnx`, a protobuf) describing a model as a dataflow **graph**: typed tensors flowing through a fixed set of standardized **operators** (`Conv`, `MatMul`, `Relu`, `Gemm`, …). It's a *specification*, not a library.
- **ONNX Runtime (ORT)** — Microsoft's high-performance *inference engine* that loads an `.onnx` graph and executes it, applying graph optimizations and dispatching to hardware-specific backends (**execution providers**: CPU, CUDA, TensorRT, CoreML, DirectML, …).

**The problem it solves.** Training frameworks (PyTorch, TensorFlow, scikit-learn) are great for *building* models but heavy and awkward for *deploying* them. You don't want PyTorch + Python + CUDA on a phone, in a C# service, or in a browser. ONNX decouples **train-time** from **run-time**: export once to a neutral format, then run with a small, fast, dependency-light engine on whatever target you have.

**Reach for it when** you've trained a model and need to (a) ship inference without dragging the training framework along, (b) run on a language/platform that isn't Python, (c) squeeze more throughput/latency out of CPU or specialized accelerators, or (d) standardize many models behind one serving interface.

**Don't reach for it when** you're still iterating on training (stay in your framework), your model uses exotic custom ops with no ONNX equivalent, or you're already happy with a framework-native server (TorchServe, TF-Serving) and don't need portability.

## 2. Mental Model

**Think of ONNX as a shipping container and ORT as the truck.** Your model gets packed once into a standard container (the graph); any compliant truck (execution provider) can carry it, and the cargo arrives identical.

```
   TRAIN (any framework)            EXPORT              RUN (anywhere)
 ┌───────────────────────┐     ┌──────────────┐    ┌──────────────────────┐
 │ PyTorch / TF / sklearn │ ──▶ │  model.onnx  │ ─▶ │     ONNX Runtime      │
 │  torch.onnx.export     │     │  (the graph: │    │  graph optimize  +    │
 │  tf2onnx / skl2onnx    │     │  ops+weights)│    │  pick execution       │
 └───────────────────────┘     └──────────────┘    │  provider (CPU/CUDA/  │
                                       │            │  TensorRT/CoreML/...) │
                                  static graph,     └──────────────────────┘
                                  typed tensors,            │
                                  versioned opset      sess.run(out, {in: x})
```

The key shift from a training framework: ONNX is a **static, ahead-of-time graph**, not eager Python. There's no control flow running in Python at inference — the whole computation is the graph, dtypes are fixed at export time, and ORT can therefore fuse/optimize the whole thing before the first call.

## 3. Key Concepts

- **Graph** — a directed acyclic graph of **nodes** (operator calls). Each node names its inputs/outputs by string; tensors flow along those names.
- **Operator (op) & opset** — ONNX defines a versioned standard library of ops. The **opset version** (e.g. opset 18) pins which op definitions apply. Exporter and runtime must both support it — opset mismatches are the #1 source of "this won't load."
- **Initializer** — a constant baked into the graph (your trained weights live here).
- **ValueInfo / typed tensors** — every input/output has a name, an element type (`float32`, `int64`, …) and a shape. Dimensions can be **dynamic** (`None` / a symbolic name like `"batch"`) so one model serves variable batch sizes or sequence lengths.
- **InferenceSession** — the ORT object that loads a model and runs it. `sess.run(output_names, {input_name: array})`; pass `output_names=None` to get all outputs.
- **Execution Provider (EP)** — a pluggable backend. ORT tries EPs in the order you list them and falls back to CPU for ops an EP can't handle. `CPUExecutionProvider` always exists.
- **Graph optimizations** — ORT rewrites the graph at load (constant folding, operator fusion, dead-node elimination) at levels `disabled`/`basic`/`extended`/`all`.
- **Quantization** — convert weights/activations to int8 (or fp16) for smaller, faster models with a small accuracy cost; a major reason teams move to ONNX for deployment.

## 4. Setup

ONNX Runtime ships as a self-contained wheel (the bundled CPU build needs no GPU, no CUDA, no training framework). You typically want three pieces: `onnx` (format + helpers), `onnxruntime` (the engine), and an **exporter** for your source framework — here `skl2onnx` for scikit-learn (PyTorch ships its own `torch.onnx.export`, TensorFlow uses `tf2onnx`).

For GPU, install `onnxruntime-gpu` instead of `onnxruntime` (don't install both — they conflict).

In [1]:
# On a fresh environment, uncomment to install (CPU build — small, no GPU needed):
# %pip install -q onnx onnxruntime skl2onnx scikit-learn

import numpy as np
import onnx
import onnxruntime as ort

print("onnx       ", onnx.__version__)
print("onnxruntime", ort.__version__)
print("providers  ", ort.get_available_providers())  # CPUExecutionProvider always present

onnx        1.22.0
onnxruntime 1.27.0
providers   ['CoreMLExecutionProvider', 'AzureExecutionProvider', 'CPUExecutionProvider']


## 5. Worked Examples

### Example 1 — Export a scikit-learn model and run it with ONNX Runtime

The canonical deployment path: train in your framework, convert to `.onnx`, run with ORT, and verify the predictions still match. Note the explicit `float32` cast — ONNX is statically typed, so you decide dtypes at export time, not at call time.

In [2]:
from sklearn.datasets import load_iris
from sklearn.linear_model import LogisticRegression
from skl2onnx import to_onnx

X, y = load_iris(return_X_y=True)
X = X.astype(np.float32)                      # ONNX is statically typed; commit to float32 now
clf = LogisticRegression(max_iter=1000).fit(X, y)

# Convert. skl2onnx traces the *fitted* estimator into an ONNX graph.
# The second arg is a sample input used to infer shapes/dtypes.
onnx_model = to_onnx(clf, X[:1])
with open("iris_logreg.onnx", "wb") as f:
    f.write(onnx_model.SerializeToString())

# Load into ORT and run inference. Inputs are passed by name as a dict.
sess = ort.InferenceSession(onnx_model.SerializeToString(),
                            providers=["CPUExecutionProvider"])
input_name = sess.get_inputs()[0].name
onnx_pred = sess.run(None, {input_name: X})[0]   # outputs: [labels, probabilities]

print("input name :", input_name)
print("sklearn[:8]:", clf.predict(X)[:8])
print("onnx   [:8]:", onnx_pred[:8])
print("agreement  :", (clf.predict(X) == onnx_pred).mean())

input name : X
sklearn[:8]: [0 0 0 0 0 0 0 0]
onnx   [:8]: [0 0 0 0 0 0 0 0]
agreement  : 1.0


### Example 2 — Build a raw ONNX graph by hand

To make the "model = graph of operators" mental model concrete, here's a complete model with no training framework at all: a linear layer `Y = X @ W + b` assembled from two nodes (`MatMul`, `Add`) plus weight initializers. This is exactly what an exporter produces under the hood, and it shows how dynamic batch dimensions and the opset are declared.

In [3]:
from onnx import helper, TensorProto, numpy_helper

# Weights as graph "initializers" (baked-in constants).
W = numpy_helper.from_array(np.array([[1.0], [2.0]], dtype=np.float32), name="W")
b = numpy_helper.from_array(np.array([0.5], dtype=np.float32), name="b")

# Inputs/outputs are typed, named tensors. None = a dynamic dimension (any batch size).
inp = helper.make_tensor_value_info("X", TensorProto.FLOAT, [None, 2])
out = helper.make_tensor_value_info("Y", TensorProto.FLOAT, [None, 1])

# Nodes wire tensors together by string name: X@W -> "XW", then XW + b -> "Y".
matmul = helper.make_node("MatMul", ["X", "W"], ["XW"])
add    = helper.make_node("Add",    ["XW", "b"], ["Y"])

graph = helper.make_graph([matmul, add], "linear", [inp], [out], initializer=[W, b])
model = helper.make_model(graph, opset_imports=[helper.make_opsetid("", 18)])
onnx.checker.check_model(model)              # validates structure, types, opset

sess = ort.InferenceSession(model.SerializeToString(), providers=["CPUExecutionProvider"])
batch = np.array([[1.0, 1.0], [3.0, 4.0]], dtype=np.float32)  # dynamic batch of 2
print("Y =\n", sess.run(None, {"X": batch})[0])              # [[3.5], [11.5]]

Y =
 [[ 3.5]
 [11.5]]


### Example 3 — Inspect a model: opset, inputs, outputs

Before deploying an `.onnx` you didn't build, inspect it. The opset, input/output names, and (possibly dynamic) shapes are the contract you must satisfy at `sess.run` time. ORT also fails loudly here if the opset is newer than the runtime supports.

In [4]:
m = onnx.load("iris_logreg.onnx")

print("opset      :", [(op.domain or "ai.onnx", op.version) for op in m.opset_import])

def shape(v):
    return [d.dim_value if d.HasField("dim_value") else (d.dim_param or "?")
            for d in v.type.tensor_type.shape.dim]

print("inputs     :", [(i.name, shape(i)) for i in m.graph.input])
print("outputs    :", [o.name for o in m.graph.output])
print("n nodes    :", len(m.graph.node))
print("op types   :", sorted({n.op_type for n in m.graph.node}))

opset      : [('ai.onnx', 9), ('ai.onnx.ml', 1)]
inputs     : [('X', ['?', 4])]
outputs    : ['output_label', 'output_probability']
n nodes    : 4
op types   : ['Cast', 'LinearClassifier', 'Normalizer', 'ZipMap']


## 6. Gotchas & Pitfalls

- **Opset mismatch.** A model exported at a newer opset than your `onnxruntime` supports won't load. Fix by upgrading ORT or exporting at a lower opset (`opset_version=` / `target_opset=`). Always pin both.
- **dtype surprises.** ONNX is statically typed. The classic bug: training in `float64` but feeding `float32` (or vice versa) at inference — ORT raises a type error, it won't silently cast. Decide dtypes at export and feed exactly that.
- **Dynamic vs fixed shapes.** If you export with a concrete batch size, the model *only* accepts that size. Mark batch/sequence dims as dynamic (`dynamic_axes=` in `torch.onnx.export`, `None` dims in raw graphs) or you'll hit shape-mismatch errors in production.
- **Unsupported / custom ops.** Exotic layers may have no ONNX op. Export fails or produces a model ORT can't run. Options: rewrite the layer with supported ops, bump the opset, or register a custom op.
- **EP fallback is silent.** If your CUDA/TensorRT EP can't handle some nodes, ORT silently runs them on CPU — you think you're on GPU but you're not. Check `sess.get_providers()` and profile.
- **Output is a list.** `sess.run` returns a *list* of arrays (one per output) even for a single output — `result[0]`, not `result`. sklearn classifiers convert to *two* outputs (labels and probabilities).
- **Numerical drift.** Optimizations, fused kernels, and quantization can shift outputs slightly. Validate parity against the source model (as in Example 1) before trusting it; don't assume bit-identical.
- **`onnxruntime` vs `onnxruntime-gpu`.** Installing both, or mixing versions, causes import/loading chaos. Pick one per environment.

## 7. When to Use vs Alternatives

| Option | Best for | Trade-off vs ONNX Runtime |
|---|---|---|
| **ONNX Runtime** | Portable, fast inference across languages/platforms; CPU + many accelerators behind one API | Inference only — not for training; some ops/models need conversion effort |
| **Native framework (PyTorch eager / TF)** | Iterating, training, research; full op coverage and Python control flow | Heavyweight to deploy; slower CPU inference; ties runtime to the framework |
| **`torch.compile` / TorchScript** | Speeding up PyTorch *inside* PyTorch with minimal export work | Stays in the PyTorch/Python world — no cross-language/edge portability |
| **TensorRT (direct)** | Squeezing maximum throughput on NVIDIA GPUs | NVIDIA-only, more setup; ORT can use TensorRT as an EP for most of the win with less lock-in |
| **TF-Serving / TorchServe** | A managed server for one framework's models | Framework-locked; ONNX gives you one format + engine for *many* source frameworks |
| **OpenVINO / CoreML / TFLite** | Vendor-tuned edge/mobile runtimes | Platform-specific; ONNX is the cross-vendor lingua franca (and ORT has EPs for several of these) |

**Rule of thumb:** train in whatever framework you like, then export to ONNX when you need deployment that is *fast*, *portable*, and *decoupled* from training. If you never leave Python/PyTorch and don't need the speed, ONNX is overhead you can skip.

## 8. Resources

- **ONNX Runtime docs** — install, EPs, optimization, quantization: https://onnxruntime.ai/docs/
- **ONNX format spec & operator list** — the IR, opsets, and every standard op: https://onnx.ai/onnx/
- **`torch.onnx` export guide** — the PyTorch → ONNX path (dynamic axes, opset, the new dynamo exporter): https://pytorch.org/docs/stable/onnx.html
- **`skl2onnx` docs** — scikit-learn → ONNX, tutorials and supported estimators: https://onnx.ai/sklearn-onnx/
- **ONNX Model Zoo** — pretrained `.onnx` models to inspect and run: https://github.com/onnx/models